In [ ]:
# ============================================
# Cell 1: 配置路径和加载数据
# ============================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Image, display

# TODO: 填入实际路径
retrieval_npy_path = "/path/to/SpecVLA/openvla/specdecoding/test-speed/libero_naive_retrieval_Exp_online_Memory/EVAL-libero_goal-base-2026_01_17-03_43_24--record_guiji_suc_fail_pure_db_observations.npy"  # 纯DB检索的observations.npy路径
sd_npy_path = "/path/to/SpecVLA/openvla/specdecoding/test-speed/libero_goal_Retrieval_Verify/EVAL-libero_goal-openvla-2026_01_17-12_50_04--record_guiji_suc_fail_pure_sd_observations.npy"         # 1:1 DB+Model的observations.npy路径

# 输出目录
output_dir = "/path/to/SpecVLA/trajectory_visualizations"
import os
os.makedirs(output_dir, exist_ok=True)

# 辅助函数：从任务描述生成文件名友好的任务名
def get_task_name_from_description(task_description):
    """将任务描述转换为文件名友好的格式"""
    # 移除标点符号，转小写，用下划线连接
    import re
    # 移除特殊字符，只保留字母数字和空格
    clean_desc = re.sub(r'[^\w\s]', '', task_description.lower())
    # 将空格替换为下划线，并限制长度
    task_name = '_'.join(clean_desc.split())
    # 限制长度（防止文件名过长）
    if len(task_name) > 50:
        task_name = task_name[:50]
    return task_name

In [ ]:
# ============================================
# Cell 2: 加载数据并解析
# ============================================
def load_observations(npy_path):
    """加载observations.npy并返回字典"""
    data = np.load(npy_path, allow_pickle=True)
    obs_dict = data.item()
    return obs_dict

def get_trajectory_info(obs_dict):
    """获取所有轨迹的信息摘要"""
    info = {}
    for task_id in obs_dict.keys():
        info[task_id] = {}
        for episode_idx in obs_dict[task_id].keys():
            episode_data = obs_dict[task_id][episode_idx]
            info[task_id][episode_idx] = {
                'success': episode_data['success'],
                'task_description': episode_data['task_description'],
                'num_steps': episode_data['num_steps']
            }
    return info

# 加载数据
print("加载数据中...")
retrieval_data = load_observations(retrieval_npy_path)
sd_data = load_observations(sd_npy_path)

print(f"检索数据 - 任务数: {len(retrieval_data)}")
print(f"SD数据 - 任务数: {len(sd_data)}")

# 获取轨迹信息
retrieval_info = get_trajectory_info(retrieval_data)
sd_info = get_trajectory_info(sd_data)

In [ ]:
# ============================================
# Cell 3: 辅助函数 - 提取轨迹坐标和计算距离
# ============================================
def extract_trajectory_xyz(episode_data):
    """从episode数据中提取xyz轨迹坐标"""
    observations = episode_data['observations']
    states = np.array([obs['state'] for obs in observations])
    x = states[:, 0]
    y = states[:, 1]
    z = states[:, 2]
    return x, y, z

def compute_point_distances(traj1_xyz, traj2_xyz):
    """
    计算两条轨迹对应点之间的距离
    返回每个时间步的距离列表
    """
    x1, y1, z1 = traj1_xyz
    x2, y2, z2 = traj2_xyz
    
    # 取较短轨迹的长度
    min_len = min(len(x1), len(x2))
    
    distances = []
    for i in range(min_len):
        dist = np.sqrt((x1[i] - x2[i])**2 + (y1[i] - y2[i])**2 + (z1[i] - z2[i])**2)
        distances.append(dist)
    
    return np.array(distances)

def find_overlapping_points(distances, threshold=0.02):
    """
    找到距离小于阈值的重叠点索引
    threshold: 距离阈值（单位：米）
    """
    return np.where(distances < threshold)[0]

print("辅助函数定义完成")

In [ ]:
# ============================================
# Cell 4: 找到符合条件的轨迹对（交叉配对版本）
# ============================================
def find_trajectory_pairs(retrieval_info, sd_info):
    """
    找到不同成功状态组合的轨迹对（同一任务内交叉配对）
    返回：
    - both_success: 两种方法都成功的 (task_id, retrieval_episode_idx, sd_episode_idx, task_desc) 列表
    - sd_success_retrieval_fail: SD成功但检索失败的列表
    - retrieval_success_sd_fail: 检索成功但SD失败的列表
    - both_fail: 两种方法都失败的列表
    """
    both_success = []
    sd_success_retrieval_fail = []
    retrieval_success_sd_fail = []
    both_fail = []
    
    for task_id in retrieval_info.keys():
        if task_id not in sd_info:
            continue
        
        # 收集该任务下不同成功状态的episodes
        retrieval_success_episodes = []
        retrieval_fail_episodes = []
        sd_success_episodes = []
        sd_fail_episodes = []
        
        task_desc = None
        
        # 分类检索结果
        for r_eid in retrieval_info[task_id].keys():
            if task_desc is None:
                task_desc = retrieval_info[task_id][r_eid]['task_description']
            if retrieval_info[task_id][r_eid]['success']:
                retrieval_success_episodes.append(r_eid)
            else:
                retrieval_fail_episodes.append(r_eid)
        
        # 分类SD结果
        for s_eid in sd_info[task_id].keys():
            if sd_info[task_id][s_eid]['success']:
                sd_success_episodes.append(s_eid)
            else:
                sd_fail_episodes.append(s_eid)
        
        # 交叉配对
        # 1. 两种方法都成功：检索成功 × SD成功
        for r_eid in retrieval_success_episodes:
            for s_eid in sd_success_episodes:
                both_success.append((task_id, r_eid, s_eid, task_desc))
        
        # 2. SD成功但检索失败：检索失败 × SD成功
        for r_eid in retrieval_fail_episodes:
            for s_eid in sd_success_episodes:
                sd_success_retrieval_fail.append((task_id, r_eid, s_eid, task_desc))
        
        # 3. 检索成功但SD失败：检索成功 × SD失败
        for r_eid in retrieval_success_episodes:
            for s_eid in sd_fail_episodes:
                retrieval_success_sd_fail.append((task_id, r_eid, s_eid, task_desc))
        
        # 4. 两种方法都失败：检索失败 × SD失败
        for r_eid in retrieval_fail_episodes:
            for s_eid in sd_fail_episodes:
                both_fail.append((task_id, r_eid, s_eid, task_desc))
    
    return both_success, sd_success_retrieval_fail, retrieval_success_sd_fail, both_fail

# 找到轨迹对（交叉配对）
both_success, sd_success_retrieval_fail, retrieval_success_sd_fail, both_fail = find_trajectory_pairs(retrieval_info, sd_info)

print(f"统计结果（交叉配对）:")
print(f"  两种方法都成功: {len(both_success)} 对")
print(f"  SD成功但检索失败: {len(sd_success_retrieval_fail)} 对")
print(f"  检索成功但SD失败: {len(retrieval_success_sd_fail)} 对")
print(f"  两种方法都失败: {len(both_fail)} 对")

# 显示部分示例
if len(both_success) > 0:
    print(f"\n两种方法都成功的示例 (前3个):")
    for i, (tid, r_eid, s_eid, desc) in enumerate(both_success[:3]):
        print(f"  Task {tid}, Retrieval Episode {r_eid} vs SD Episode {s_eid}: {desc[:50]}...")

if len(sd_success_retrieval_fail) > 0:
    print(f"\nSD成功但检索失败的示例 (前3个):")
    for i, (tid, r_eid, s_eid, desc) in enumerate(sd_success_retrieval_fail[:3]):
        print(f"  Task {tid}, Retrieval Episode {r_eid} vs SD Episode {s_eid}: {desc[:50]}...")

In [ ]:
# ============================================
# Cell 5: 可视化函数 - 绘制双轨迹对比图
# ============================================
def plot_trajectory_comparison(retrieval_xyz, sd_xyz, 
                                retrieval_success, sd_success,
                                task_description, task_id, episode_idx,
                                overlap_threshold=0.02,
                                save_path=None):
    """
    绘制两条轨迹的对比图
    
    参数:
    - retrieval_xyz: (x, y, z) 检索轨迹坐标
    - sd_xyz: (x, y, z) SD轨迹坐标
    - retrieval_success: 检索是否成功
    - sd_success: SD是否成功
    - overlap_threshold: 判断重叠的距离阈值
    - save_path: 保存路径
    """
    r_x, r_y, r_z = retrieval_xyz
    s_x, s_y, s_z = sd_xyz
    
    # 创建图
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # ========== 绘制检索轨迹 (蓝色) ==========
    ax.plot(r_x, r_y, r_z, 'b-', linewidth=1.5, alpha=0.5)
    ax.scatter(r_x, r_y, r_z, c='royalblue', s=60, alpha=0.7, 
               edgecolors='none', label='Retrieval')
    
    # ========== 绘制SD轨迹 (橙色) ==========
    ax.plot(s_x, s_y, s_z, color='darkorange', linewidth=1.5, alpha=0.5)
    ax.scatter(s_x, s_y, s_z, c='darkorange', s=60, alpha=0.7, 
               edgecolors='none', label='SD 1:1')
    
    # ========== 起点 (分别标注) ==========
    # 检索起点 (蓝色圆形)
    ax.scatter(r_x[0], r_y[0], r_z[0], c='royalblue', s=200, marker='o', 
               edgecolors='black', linewidth=2.5, label='Retrieval Start', zorder=10)
    # SD起点 (橙色圆形)
    ax.scatter(s_x[0], s_y[0], s_z[0], c='darkorange', s=200, marker='o', 
               edgecolors='black', linewidth=2.5, label='SD Start', zorder=10)
    
    # ========== 终点区分 ==========
    # 检索终点 (蓝色方形)
    ax.scatter(r_x[-1], r_y[-1], r_z[-1], c='royalblue', s=200, marker='s', 
               edgecolors='black', linewidth=2.5, label='Retrieval End', zorder=10)
    # SD终点 (橙色三角)
    ax.scatter(s_x[-1], s_y[-1], s_z[-1], c='darkorange', s=200, marker='^', 
               edgecolors='black', linewidth=2.5, label='SD End', zorder=10)
    
    # ========== 图表设置 ==========
    # 轴标签（已移除）
    # ax.set_xlabel('X', fontsize=32, labelpad=20, fontweight='bold', fontname='Arial')
    # ax.set_ylabel('Y', fontsize=32, labelpad=20, fontweight='bold', fontname='Arial')
    # ax.set_zlabel('Z', fontsize=32, labelpad=20, fontweight='bold', fontname='Arial')
    
    # 隐藏刻度数字
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    
    # 标题（已移除）
    # status_str = f"Retrieval: {'✓' if retrieval_success else '✗'} | SD: {'✓' if sd_success else '✗'}"
    # title = f"Task {task_id}, Episode {episode_idx}\n{task_description[:60]}...\n[{status_str}]"
    # ax.set_title(title, fontsize=11, pad=20)
    
    # 图例（已移除）
    # ax.legend(loc='upper right', fontsize=12, framealpha=0.9, edgecolor='black')
    
    # 网格和视角
    ax.grid(True, alpha=0.3)
    ax.view_init(elev=25, azim=45)
    
    plt.tight_layout()
    
    # 保存
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"图片已保存: {save_path}")
    
    plt.close()
    return save_path

print("可视化函数定义完成")

In [ ]:
# ============================================
# Cell 6: 绘制 - 两种方法都成功的轨迹对比
# ============================================
print("="*60)
print("绘制两种方法都成功的轨迹对比图")
print("="*60)

if len(both_success) > 0:
    # 选择第一对成功的轨迹
    task_id, r_eid, s_eid, task_desc = both_success[0]
    
    print(f"\n选择: Task {task_id}, Retrieval Episode {r_eid} vs SD Episode {s_eid}")
    print(f"任务描述: {task_desc}")
    
    # 提取轨迹
    r_episode = retrieval_data[task_id][r_eid]
    s_episode = sd_data[task_id][s_eid]
    
    r_xyz = extract_trajectory_xyz(r_episode)
    s_xyz = extract_trajectory_xyz(s_episode)
    
    print(f"检索轨迹长度: {len(r_xyz[0])} steps")
    print(f"SD轨迹长度: {len(s_xyz[0])} steps")
    
    # 计算距离统计
    distances = compute_point_distances(r_xyz, s_xyz)
    overlap_count = len(find_overlapping_points(distances, threshold=0.02))
    
    print(f"\n轨迹距离统计:")
    print(f"  平均距离: {np.mean(distances):.4f} m")
    print(f"  最大距离: {np.max(distances):.4f} m")
    print(f"  最小距离: {np.min(distances):.4f} m")
    print(f"  重叠点数 (阈值0.02m): {overlap_count}")
    
    # 生成文件名和路径（按任务名-情况结构）
    task_name = get_task_name_from_description(task_desc)
    task_folder = os.path.join(output_dir, task_name)
    category_folder = os.path.join(task_folder, "both_success")
    os.makedirs(category_folder, exist_ok=True)
    
    filename = f"{task_name}_both_success_r{r_eid}_s{s_eid}.png"
    save_path = os.path.join(category_folder, filename)
    
    # 绘图
    plot_trajectory_comparison(
        r_xyz, s_xyz,
        retrieval_success=True, sd_success=True,
        task_description=task_desc,
        task_id=task_id, episode_idx=f"R{r_eid}_S{s_eid}",
        overlap_threshold=0.02,
        save_path=save_path
    )
    
    # 显示图片
    display(Image(filename=save_path))
else:
    print("没有找到两种方法都成功的轨迹对")

In [ ]:
# ============================================
# Cell 7: 绘制 - SD成功但检索失败的轨迹对比
# ============================================
print("="*60)
print("绘制SD成功但检索失败的轨迹对比图")
print("="*60)

if len(sd_success_retrieval_fail) > 0:
    # 选择第一对
    task_id, r_eid, s_eid, task_desc = sd_success_retrieval_fail[0]
    
    print(f"\n选择: Task {task_id}, Retrieval Episode {r_eid} vs SD Episode {s_eid}")
    print(f"任务描述: {task_desc}")
    
    # 提取轨迹
    r_episode = retrieval_data[task_id][r_eid]
    s_episode = sd_data[task_id][s_eid]
    
    r_xyz = extract_trajectory_xyz(r_episode)
    s_xyz = extract_trajectory_xyz(s_episode)
    
    print(f"检索轨迹长度: {len(r_xyz[0])} steps (失败)")
    print(f"SD轨迹长度: {len(s_xyz[0])} steps (成功)")
    
    # 计算距离统计
    distances = compute_point_distances(r_xyz, s_xyz)
    overlap_count = len(find_overlapping_points(distances, threshold=0.02))
    
    print(f"\n轨迹距离统计:")
    print(f"  平均距离: {np.mean(distances):.4f} m")
    print(f"  最大距离: {np.max(distances):.4f} m")
    print(f"  最小距离: {np.min(distances):.4f} m")
    print(f"  重叠点数 (阈值0.02m): {overlap_count}")
    
    # 生成文件名和路径（按任务名-情况结构）
    task_name = get_task_name_from_description(task_desc)
    task_folder = os.path.join(output_dir, task_name)
    category_folder = os.path.join(task_folder, "sd_success_retrieval_fail")
    os.makedirs(category_folder, exist_ok=True)
    
    filename = f"{task_name}_sd_success_retrieval_fail_r{r_eid}_s{s_eid}.png"
    save_path = os.path.join(category_folder, filename)
    
    # 绘图
    plot_trajectory_comparison(
        r_xyz, s_xyz,
        retrieval_success=False, sd_success=True,
        task_description=task_desc,
        task_id=task_id, episode_idx=f"R{r_eid}_S{s_eid}",
        overlap_threshold=0.02,
        save_path=save_path
    )
    
    # 显示图片
    display(Image(filename=save_path))
else:
    print("没有找到SD成功但检索失败的轨迹对")

In [ ]:
# ============================================
# Cell 8: 批量绘制多个对比图
# ============================================
def batch_plot_comparisons(trajectory_pairs, retrieval_data, sd_data, 
                           category_name, category_short_name, max_plots=5):
    """批量绘制多个轨迹对比图"""
    print(f"\n{'='*60}")
    print(f"批量绘制: {category_name}")
    print(f"{'='*60}")
    
    if len(trajectory_pairs) == 0:
        print(f"没有找到 {category_name} 的轨迹对")
        return []
    
    saved_paths = []
    num_to_plot = min(len(trajectory_pairs), max_plots)
    
    for i, (task_id, r_eid, s_eid, task_desc) in enumerate(trajectory_pairs[:num_to_plot]):
        print(f"\n[{i+1}/{num_to_plot}] Task {task_id}, Retrieval Episode {r_eid} vs SD Episode {s_eid}")
        
        # 提取轨迹
        r_episode = retrieval_data[task_id][r_eid]
        s_episode = sd_data[task_id][s_eid]
        
        r_xyz = extract_trajectory_xyz(r_episode)
        s_xyz = extract_trajectory_xyz(s_episode)
        
        r_success = retrieval_info[task_id][r_eid]['success']
        s_success = sd_info[task_id][s_eid]['success']
        
        # 生成文件名和路径
        task_name = get_task_name_from_description(task_desc)
        
        # 按任务名创建文件夹，然后在其下创建情况子文件夹
        task_folder = os.path.join(output_dir, task_name)
        category_folder = os.path.join(task_folder, category_short_name)
        os.makedirs(category_folder, exist_ok=True)
        
        # 文件名格式：任务名_情况_r检索episode_s模型episode.png
        filename = f"{task_name}_{category_short_name}_r{r_eid}_s{s_eid}.png"
        save_path = os.path.join(category_folder, filename)
        
        # 绘图
        plot_trajectory_comparison(
            r_xyz, s_xyz,
            retrieval_success=r_success, sd_success=s_success,
            task_description=task_desc,
            task_id=task_id, episode_idx=f"R{r_eid}_S{s_eid}",
            overlap_threshold=0.02,
            save_path=save_path
        )
        saved_paths.append(save_path)
    
    return saved_paths

# 批量绘制各类别的图
print("\n开始批量绘制...")

# 1. 两种方法都成功
paths_both_success = batch_plot_comparisons(
    both_success, retrieval_data, sd_data, 
    "SD Success & Retrieval Success", "both_success", max_plots=100
)

# 2. SD成功但检索失败
paths_sd_success = batch_plot_comparisons(
    sd_success_retrieval_fail, retrieval_data, sd_data, 
    "SD Success & Retrieval Fail", "sd_success_retrieval_fail", max_plots=100
)

# 3. 检索成功但SD失败
paths_retrieval_success = batch_plot_comparisons(
    retrieval_success_sd_fail, retrieval_data, sd_data, 
    "SD Fail & Retrieval Success", "sd_fail_retrieval_success", max_plots=100
)

# 4. 两种方法都失败
paths_both_fail = batch_plot_comparisons(
    both_fail, retrieval_data, sd_data, 
    "SD Fail & Retrieval Fail", "both_fail", max_plots=100
)

print(f"\n\n{'='*60}")
print("批量绘制完成!")
print(f"图片保存目录: {output_dir}")
print(f"{'='*60}")

In [ ]:
# ============================================
# Cell 9: 总体统计和汇总
# ============================================
def compute_overall_statistics(trajectory_pairs, retrieval_data, sd_data, category_name):
    """计算某类轨迹对的总体统计"""
    if len(trajectory_pairs) == 0:
        return None
    
    all_avg_distances = []
    all_overlap_ratios = []
    
    for task_id, r_eid, s_eid, _ in trajectory_pairs:
        r_episode = retrieval_data[task_id][r_eid]
        s_episode = sd_data[task_id][s_eid]
        
        r_xyz = extract_trajectory_xyz(r_episode)
        s_xyz = extract_trajectory_xyz(s_episode)
        
        distances = compute_point_distances(r_xyz, s_xyz)
        overlap_indices = find_overlapping_points(distances, threshold=0.02)
        
        all_avg_distances.append(np.mean(distances))
        overlap_ratio = len(overlap_indices) / len(distances) if len(distances) > 0 else 0
        all_overlap_ratios.append(overlap_ratio)
    
    return {
        'category': category_name,
        'count': len(trajectory_pairs),
        'avg_distance_mean': np.mean(all_avg_distances),
        'avg_distance_std': np.std(all_avg_distances),
        'overlap_ratio_mean': np.mean(all_overlap_ratios) * 100,
        'overlap_ratio_std': np.std(all_overlap_ratios) * 100
    }

# 计算统计
print("="*70)
print("总体统计汇总")
print("="*70)

stats_list = []

stats = compute_overall_statistics(both_success, retrieval_data, sd_data, "SD Success & Retrieval Success")
if stats:
    stats_list.append(stats)

stats = compute_overall_statistics(sd_success_retrieval_fail, retrieval_data, sd_data, "SD Success & Retrieval Fail")
if stats:
    stats_list.append(stats)

stats = compute_overall_statistics(retrieval_success_sd_fail, retrieval_data, sd_data, "SD Fail & Retrieval Success")
if stats:
    stats_list.append(stats)

stats = compute_overall_statistics(both_fail, retrieval_data, sd_data, "SD Fail & Retrieval Fail")
if stats:
    stats_list.append(stats)

# 打印统计表
print(f"\n{'Category':<40} {'Count':>8} {'Avg Dist (m)':>15} {'Overlap %':>12}")
print("-"*70)
for s in stats_list:
    print(f"{s['category']:<40} {s['count']:>8} {s['avg_distance_mean']:>10.4f}±{s['avg_distance_std']:.4f} {s['overlap_ratio_mean']:>8.1f}±{s['overlap_ratio_std']:.1f}%")

print("\n" + "="*70)
print("分析结论:")
print("="*70)
print("- 两种方法都成功时，轨迹重叠度高说明两种方法产生了相似的路径")
print("- SD成功但检索失败时，可以观察SD如何纠正错误的检索结果")
print("- 检索成功但SD失败时，说明检索效果好但模型生成能力不足")
print("- 两种方法都失败时，可能是任务本身难度较大")

# 保存统计结果到文件
analysis_file = os.path.join(output_dir, "Analysis.txt")
with open(analysis_file, 'w', encoding='utf-8') as f:
    f.write("="*70 + "\n")
    f.write("总体统计汇总\n")
    f.write("="*70 + "\n\n")
    
    f.write(f"{'Category':<40} {'Count':>8} {'Avg Dist (m)':>15} {'Overlap %':>12}\n")
    f.write("-"*70 + "\n")
    for s in stats_list:
        f.write(f"{s['category']:<40} {s['count']:>8} {s['avg_distance_mean']:>10.4f}±{s['avg_distance_std']:.4f} {s['overlap_ratio_mean']:>8.1f}±{s['overlap_ratio_std']:.1f}%\n")
    
    f.write("\n" + "="*70 + "\n")
    f.write("分析结论:\n")
    f.write("="*70 + "\n")
    f.write("- 两种方法都成功时，轨迹重叠度高说明两种方法产生了相似的路径\n")
    f.write("- SD成功但检索失败时，可以观察SD如何纠正错误的检索结果\n")
    f.write("- 检索成功但SD失败时，说明检索效果好但模型生成能力不足\n")
    f.write("- 两种方法都失败时，可能是任务本身难度较大\n")
    
    f.write("\n" + "="*70 + "\n")
    f.write("详细统计:\n")
    f.write("="*70 + "\n")
    f.write(f"SD Success & Retrieval Success: {len(both_success)} 对\n")
    f.write(f"SD Success & Retrieval Fail: {len(sd_success_retrieval_fail)} 对\n")
    f.write(f"SD Fail & Retrieval Success: {len(retrieval_success_sd_fail)} 对\n")
    f.write(f"SD Fail & Retrieval Fail: {len(both_fail)} 对\n")
    f.write(f"总计: {len(both_success) + len(sd_success_retrieval_fail) + len(retrieval_success_sd_fail) + len(both_fail)} 对\n")

print(f"\n统计结果已保存到: {analysis_file}")

In [ ]:
# ============================================
# Cell 10: 显示保存的所有图片
# ============================================
print("保存的图片列表:")
print("="*60)

import glob
# 递归搜索所有子文件夹中的png文件
all_images = glob.glob(os.path.join(output_dir, "**/*.png"), recursive=True)
all_images.sort()

# 按任务名和情况分组显示
from collections import defaultdict
images_by_task = defaultdict(lambda: defaultdict(list))

for img_path in all_images:
    # 解析路径: output_dir/task_name/category/filename.png
    rel_path = os.path.relpath(img_path, output_dir)
    parts = rel_path.split(os.sep)
    if len(parts) >= 3:
        task_name = parts[0]
        category = parts[1]
        images_by_task[task_name][category].append(img_path)

# 显示统计
total_images = 0
for task_name in sorted(images_by_task.keys()):
    print(f"\n【{task_name}】")
    for category in sorted(images_by_task[task_name].keys()):
        count = len(images_by_task[task_name][category])
        total_images += count
        print(f"  {category}: {count} 张")
        # 显示文件名
        for img_path in images_by_task[task_name][category]:
            print(f"    - {os.path.basename(img_path)}")

print(f"\n{'='*60}")
print(f"总计: {total_images} 张图片")
print(f"任务数: {len(images_by_task)}")
print(f"保存目录: {output_dir}")
print(f"{'='*60}")

# 可选: 显示所有图片
# for img_path in all_images:
#     print(f"\n{os.path.basename(img_path)}:")
#     display(Image(filename=img_path))